In [59]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
# Load dataset into a DataFrame
df = pd.read_csv('./world_happiness_2023.csv')
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']


print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
print(df.head())

In [ ]:
# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())
print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())
print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])

## **Task 1 — Regional Comparison Bar Chart**

In [ ]:
# Compute aggregated statistics
region_avg = (df.groupby('Region')['Happiness_Score']
              .mean()
              .reset_index()
              .sort_values('Happiness_Score'))  # sort for horizontal bar
region_avg

# Plot Chart
fig1 = px.bar(
    region_avg,
    x='Happiness_Score',
    y = 'Region',
    orientation='h',
    color='Happiness_Score',
    color_continuous_scale='Viridis'
)

fig1.update_layout(
    title='Nordic countries lead global happiness — strong social support systems make the difference',
    xaxis_title='Average Happiness Score',
    yaxis_title='Region',
    xaxis=dict(range=[0, region_avg['Happiness_Score'].max() + 1]),
    template="plotly_white"
)

fig1.show()

### **Key Insights:**

*   Western Europe stands out as the happiest region globally. This matters because it highlights the impact of strong social safety nets, high GDP per capita, and institutional trust on overall life satisfaction.
*   In contrast, regions with lower scores may reflect economic instability, weaker public services, or social challenges, emphasizing the importance of policy and development in improving well-being.




# **Task 2: Top 8 vs. Bottom 8 contrast**

In [ ]:
# Get top and bottom countries
top_n = 8

top8 = df.nlargest(top_n, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'
bottom8 = df.nsmallest(top_n, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score')

# Plot Chart
fig2 = px.bar(
    combined,
    x='Happiness_Score',
    y='Country',
    color='Group',
    orientation='h',
    color_discrete_map={'Top 8': '#2a9d8f', 'Bottom 8': '#e76f51'}
)

# Layout Improvements
fig2.update_traces(
    marker_line_width=1
)

fig2.update_layout(
    title= "<b>A Stark Global Divide:</b> Top Countries Are Nearly Twice as Happy as the Lowest-Ranked",
    xaxis_title='Happiness Score',
    yaxis_title='Country',
    plot_bgcolor='white'
)

# Horizontal Seperator
separator_y = top_n - 0.5

fig2.add_shape(
    type="line",
    x0=0, x1=1, xref="paper",
    y0=separator_y, y1=separator_y, yref="y",
    line=dict(color="grey", width=2, dash="dash")
)

# Global Average Line
global_avg = df['Happiness_Score'].mean()

fig2.add_vline(
    x=global_avg,
    line_width=2,
    line_dash="dash",
    line_color="#000000",
    annotation_text=f"  Global Avg. ({global_avg:.2f})",
    annotation_position="top top",
    annotation_font=dict(size=13, color="#264653")
)

fig2.show()

### **Key Insights**

*   The chart reveals a significant happiness divide between the top and bottom ranked countries.
*   The top 8 countries consistently outperform the global average, suggesting strong social systems, economic stability, and trust in institutions.
* In contrast, the bottom 8 fall well below the global mean, highlighting disparities in development, governance, and quality of life.
* This gap emphasizes that global happiness is highly uneven and strongly shaped by structural factors rather than individual differences alone.

### **Task 3 — GDP vs Freedom Across Regions**

In [ ]:
# List of Required Regions
regions=[
    'Western Europe',
    'Latin America',
    'East Asia',
    'Sub-Saharan Africa',
    'South Asia'
]

# Filter dataset to include only selected regions
df_filtered=df[df['Region'].isin(regions)]
df_filtered

In [ ]:
# Compute Regional Averages
region_summary = df_filtered.groupby('Region')[['GDP', 'Freedom']].mean().reset_index()
region_summary

In [ ]:
# Reshape Data
df_long = region_summary.melt(
    id_vars='Region',
    value_vars=['GDP', 'Freedom'],
    var_name='Indicator',
    value_name='Value'
)

df_long

In [ ]:
# Plot Grouped Bar Chart
fig3 = px.bar(
    df_long,
    x='Region',
    y='Value',
    color='Indicator',
    barmode='group',
    color_discrete_map={
        'GDP': '#e76f51',
        'Freedom': '#2a9d8f'
    }
)

# Layout Improvement
fig3.update_layout(
    title='Economic wealth strongly aligns with freedom, but gaps remain across global regions',
    xaxis_title='Region',
    yaxis_title='Average Value',
    template="plotly_white",
    yaxis=dict(range=[0, df_long['Value'].max() * 1.2])
)

# Add value labels on bars
fig3.update_traces(text=df_long['Value'].round(2), textposition='outside')

fig3.show()

### **Key Insights**
*   Western Europe consistently leads in both GDP per capita and freedom, showing a strong link between economic prosperity and personal liberties.
*   East Asia demonstrates high economic performance but relatively lower freedom compared to its GDP level, indicating a development model focused more on economic growth.
* Sub-Saharan Africa and South Asia, on the other hand, demonstrate lower scores in both indicators which indicate structural issues impacting the economic output and institutional freedom.
* This suggests that while GDP and freedom are related, the relationship is not perfectly proportional across regions.

